# DINOv2 Baseline

 image-only baseline notebook. Uses train-only CUI vocabulary with minimum frequency 5 and maximum vocabulary cap 2048 for evaluation. Retrieval ranking uses visual similarity only; CUI labels are used only after ranking for metrics.

In [ ]:
import os, re, json, math, random, time, warnings, shutil, subprocess, sys
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

try:
    import timm
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "timm"])
    import timm

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.sparse import csr_matrix

warnings.filterwarnings("ignore")


DATA_ROOT = os.environ.get("DATA_ROOT", "../data/roco_v2")


BACKBONE_PRESETS = {
    "dinov2_small_reg_full": {
        "model_name": "vit_small_patch14_reg4_dinov2.lvd142m",
        "image_size": 224,
        "batch_size": 8,
        "eval_batch_size": 16,
        "projection_dim": 512,
    },
    "dinov2_small_full": {
        "model_name": "vit_small_patch14_dinov2.lvd142m",
        "image_size": 224,
        "batch_size": 8,
        "eval_batch_size": 16,
        "projection_dim": 512,
    },
}

CFG = {
    "seed": 42,
    "random_seeds": [42, 123, 2025],
    "backbone_key": "dinov2_small_reg_full",
    "pretrained": True,


    "epochs": 4,
    "backbone_lr": 1e-5,
    "head_lr": 2e-4,
    "weight_decay": 1e-4,
    "head_dropout": 0.20,
    "grad_accum_steps": 4,
    "grad_clip_norm": 1.0,
    "gradient_checkpointing": True,


    "max_train_samples": None,
    "max_valid_samples": None,

    "num_workers": 2,
    "max_train_cuis": 2048,
    "min_cui_freq": 5,
    "eval_split": "test",
    "retrieval_pool": "test",
    "ks_main": [5, 10, 50],
    "ks_curve": list(range(1, 51)),
    "relevance_threshold": 0.0,
    "eval_chunk_size": 256,
    "topk_max": 50,
    "train": False,
    "amp": True,
    "use_trained_projection_for_retrieval": False,
}

CFG.update(BACKBONE_PRESETS[CFG["backbone_key"]])
SAFE_BACKBONE_NAME = re.sub(r"[^A-Za-z0-9_.-]+", "_", CFG["model_name"])
OUT_DIR = os.environ.get("OUTPUT_DIR", os.path.join(os.environ.get("OUTPUT_ROOT", "../results"), "dinov2_roco_full_finetune_retrieval"))
os.makedirs(OUT_DIR, exist_ok=True)
FIG_DIR = os.path.join(OUT_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

print("Selected backbone:", CFG["backbone_key"])
print("timm model:", CFG["model_name"])
print("Full fine-tuning backbone:", True)
print("Output:", OUT_DIR)


def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

seed_everything(CFG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

try:
    from IPython.display import display
except Exception:
    display = lambda x: print(x.to_string() if hasattr(x, "to_string") else x)

def savefig(name):
    path = os.path.join(FIG_DIR, name)
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved:", path)


def find_data_root(preferred):
    if Path(preferred).exists():
        return preferred
    print("DATA_ROOT not found. Searching DATA_SEARCH_DIR ...")
    candidates = []
    for root, dirs, files in os.walk(os.environ.get("DATA_SEARCH_DIR", "../data")):
        if {"train_concepts.csv", "valid_concepts.csv", "test_concepts.csv"}.issubset(set(files)):
            candidates.append(root)
    if not candidates:
        raise FileNotFoundError("Could not find folder with train/valid/test_concepts.csv")
    return candidates[0]

DATA_ROOT = find_data_root(DATA_ROOT)
print("Using DATA_ROOT:", DATA_ROOT)

CUI_RE = re.compile(r"C\d{7}", re.IGNORECASE)
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

def read_csv_flexible(path):
    for sep in [",", "\t", ";", "|"]:
        try:
            df = pd.read_csv(path, sep=sep, dtype=str, keep_default_na=False)
            if df.shape[1] >= 1:
                return df
        except Exception:
            pass
    return pd.read_csv(path, dtype=str, keep_default_na=False)

def build_image_map(split):
    split_dir = Path(DATA_ROOT) / split
    mp = {}
    for p in split_dir.rglob("*"):
        if p.suffix.lower() in IMG_EXTS:
            mp[p.stem] = str(p)
            mp[p.name] = str(p)
    return mp

def infer_id_col(df):
    for word in ["image", "file", "filename", "name", "id", "uid"]:
        for c in df.columns:
            if word in c.lower():
                return c
    return df.columns[0]

def extract_cuis_from_row(row):
    txt = " ".join([str(x) for x in row.values])
    return sorted(set([x.upper() for x in CUI_RE.findall(txt)]))

def load_split(split):
    csv_path = Path(DATA_ROOT) / f"{split}_concepts.csv"
    manual_path = Path(DATA_ROOT) / f"{split}_concepts_manual.csv"
    path = csv_path if csv_path.exists() else manual_path
    if not path.exists():
        raise FileNotFoundError(f"No concept CSV found for split={split}")

    df_raw = read_csv_flexible(path)
    id_col = infer_id_col(df_raw)
    img_map = build_image_map(split)

    rows, missing_path, no_cui = [], 0, 0
    for _, row in df_raw.iterrows():
        raw_id = str(row[id_col]).strip()
        base = Path(raw_id).stem

        img_path = None
        for key in [raw_id, base, raw_id + ".jpg", base + ".jpg", raw_id + ".png", base + ".png"]:
            if key in img_map:
                img_path = img_map[key]
                break

        if img_path is None:
            missing_path += 1
            continue

        cuis = extract_cuis_from_row(row)
        if len(cuis) == 0:
            no_cui += 1

        rows.append({"image_id": base, "path": img_path, "cuis": cuis, "n_cuis": len(cuis)})

    out = pd.DataFrame(rows)
    print(f"{split}: raw={len(df_raw)} usable={len(out)} missing_path={missing_path} no_cui={no_cui}")
    return out

train_df = load_split("train")
valid_df = load_split("valid")
test_df = load_split("test")

train_df = train_df[train_df["n_cuis"] > 0].reset_index(drop=True)
valid_df = valid_df[valid_df["n_cuis"] > 0].reset_index(drop=True)
test_df = test_df[test_df["n_cuis"] > 0].reset_index(drop=True)

overview = pd.DataFrame({
    "split": ["train", "valid", "test"],
    "images": [len(train_df), len(valid_df), len(test_df)],
    "mean_cuis_per_image": [train_df.n_cuis.mean(), valid_df.n_cuis.mean(), test_df.n_cuis.mean()],
    "median_cuis_per_image": [train_df.n_cuis.median(), valid_df.n_cuis.median(), test_df.n_cuis.median()],
})
overview.to_csv(os.path.join(OUT_DIR, "dataset_overview.csv"), index=False)
display(overview)


plt.figure(figsize=(6,4))
plt.bar(overview["split"], overview["images"])
plt.title("Images per split")
plt.ylabel("Number of images")
savefig("01_images_per_split.png")

plt.figure(figsize=(6,4))
for name, df in [("train", train_df), ("valid", valid_df), ("test", test_df)]:
    plt.hist(df["n_cuis"].values, bins=30, alpha=0.5, label=name)
plt.title("CUI count per image")
plt.xlabel("Number of CUIs")
plt.ylabel("Images")
plt.legend()
savefig("02_cui_count_histogram.png")

train_cui_counts = Counter(c for xs in train_df["cuis"] for c in xs)
top_cuis_df = pd.DataFrame(train_cui_counts.most_common(30), columns=["CUI", "count"])
top_cuis_df.to_csv(os.path.join(OUT_DIR, "top_30_train_cuis.csv"), index=False)

plt.figure(figsize=(10,5))
plt.bar(top_cuis_df["CUI"], top_cuis_df["count"])
plt.title("Top 30 CUI frequencies in training set")
plt.xticks(rotation=75, ha="right")
plt.ylabel("Frequency")
savefig("03_top_cui_frequencies.png")


def make_label_space(train_df, max_labels, min_freq):
    counts = Counter(c for xs in train_df["cuis"] for c in xs)
    labels = [c for c, n in counts.most_common() if n >= min_freq][:max_labels]
    label_to_idx = {c: i for i, c in enumerate(labels)}
    idx_to_label = {i: c for c, i in label_to_idx.items()}
    return label_to_idx, idx_to_label

label_to_idx, idx_to_label = make_label_space(train_df, CFG["max_train_cuis"], CFG["min_cui_freq"])
n_labels = len(label_to_idx)
print("Training CUI label count:", n_labels)

with open(os.path.join(OUT_DIR, "label_to_idx.json"), "w") as f:
    json.dump(label_to_idx, f, indent=2)

def to_label_indices(cuis):
    return [label_to_idx[c] for c in cuis if c in label_to_idx]

for df in [train_df, valid_df, test_df]:
    df["label_indices"] = df["cuis"].apply(to_label_indices)

train_cls_df = train_df[train_df["label_indices"].apply(len) > 0].reset_index(drop=True)
valid_cls_df = valid_df[valid_df["label_indices"].apply(len) > 0].reset_index(drop=True)
print("Train classifier samples:", len(train_cls_df))
print("Valid classifier samples:", len(valid_cls_df))


IMG_MEAN = [0.485, 0.456, 0.406]
IMG_STD = [0.229, 0.224, 0.225]

train_tfms = T.Compose([
    T.Resize((int(CFG["image_size"] * 1.15), int(CFG["image_size"] * 1.15))),
    T.RandomResizedCrop(CFG["image_size"], scale=(0.80, 1.0)),
    T.RandomRotation(degrees=5),
    T.ToTensor(),
    T.Normalize(mean=IMG_MEAN, std=IMG_STD),
])

eval_tfms = T.Compose([
    T.Resize((int(CFG["image_size"] * 1.15), int(CFG["image_size"] * 1.15))),
    T.CenterCrop(CFG["image_size"]),
    T.ToTensor(),
    T.Normalize(mean=IMG_MEAN, std=IMG_STD),
])

class FullTuneCuiDataset(Dataset):
    def __init__(self, df, transform, n_labels):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.n_labels = n_labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["path"]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        y = torch.zeros(self.n_labels, dtype=torch.float32)
        for j in row["label_indices"]:
            y[j] = 1.0
        return img, y

class RocoImageOnlyDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["path"]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, row["image_id"]


def maybe_sample_df(df, max_samples, seed):
    if max_samples is None or len(df) <= max_samples:
        return df.reset_index(drop=True)
    return df.sample(n=max_samples, random_state=seed).reset_index(drop=True)

train_cls_df = maybe_sample_df(train_cls_df, CFG["max_train_samples"], CFG["seed"])
valid_cls_df = maybe_sample_df(valid_cls_df, CFG["max_valid_samples"], CFG["seed"])
print("Train classifier samples after optional cap:", len(train_cls_df))
print("Valid classifier samples after optional cap:", len(valid_cls_df))


class TimmFeatureBackbone(nn.Module):
    def __init__(self, model_name, pretrained=True, image_size=224):
        super().__init__()


        create_kwargs = dict(pretrained=pretrained, num_classes=0)
        try:
            self.model = timm.create_model(model_name, img_size=image_size, **create_kwargs)
        except TypeError:
            self.model = timm.create_model(model_name, **create_kwargs)
        except RuntimeError as e:
            print("Pretrained loading failed with:")
            print(e)
            print("Retrying with pretrained=False so the notebook can continue.")
            try:
                self.model = timm.create_model(model_name, pretrained=False, num_classes=0, img_size=image_size)
            except TypeError:
                self.model = timm.create_model(model_name, pretrained=False, num_classes=0)

        self.num_features = int(
            getattr(self.model, "num_features", 0)
            or getattr(self.model, "embed_dim", 0)
            or getattr(getattr(self.model, "head", None), "in_features", 0)
        )
        if self.num_features <= 0:
            raise ValueError("Could not infer backbone feature dimension")

    def _pool_features(self, feat):
        if isinstance(feat, (tuple, list)):
            feat = feat[0]
        if isinstance(feat, dict):
            for key in ["x_norm_clstoken", "cls_token", "pooler_output", "image_embeds", "features"]:
                if key in feat and feat[key] is not None:
                    feat = feat[key]
                    break
            else:
                if "last_hidden_state" in feat and feat["last_hidden_state"] is not None:
                    feat = feat["last_hidden_state"]
                elif "x_norm_patchtokens" in feat and feat["x_norm_patchtokens"] is not None:
                    feat = feat["x_norm_patchtokens"]
                else:
                    feat = next(v for v in feat.values() if torch.is_tensor(v))
        if feat.ndim == 3:
            feat = feat[:, 0]
        elif feat.ndim == 4:
            feat = feat.mean(dim=(-2, -1))
        return feat.float()

    def forward(self, x):
        return self._pool_features(self.model(x))

class CUIProjectionHead(nn.Module):
    def __init__(self, in_dim, proj_dim, n_labels, dropout=0.2):
        super().__init__()
        self.norm = nn.LayerNorm(in_dim)
        self.projector = nn.Sequential(
            nn.Linear(in_dim, proj_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(proj_dim, proj_dim),
        )
        self.classifier = nn.Linear(proj_dim, n_labels)

    def forward(self, x, return_embedding=False):
        z = self.projector(self.norm(x))
        z = F.normalize(z, dim=1)
        if return_embedding:
            return z
        return self.classifier(z)

class FullDINOv2RetrievalModel(nn.Module):
    def __init__(self, model_name, pretrained, image_size, proj_dim, n_labels, dropout):
        super().__init__()
        self.backbone = TimmFeatureBackbone(model_name, pretrained=pretrained, image_size=image_size)
        self.head = CUIProjectionHead(self.backbone.num_features, proj_dim, n_labels, dropout=dropout)

    def forward(self, x, return_embedding=False):
        feat = self.backbone(x)
        return self.head(feat, return_embedding=return_embedding)

model = FullDINOv2RetrievalModel(
    model_name=CFG["model_name"],
    pretrained=CFG["pretrained"],
    image_size=CFG["image_size"],
    proj_dim=CFG["projection_dim"],
    n_labels=n_labels,
    dropout=CFG["head_dropout"],
).to(device)

if CFG["gradient_checkpointing"] and hasattr(model.backbone.model, "set_grad_checkpointing"):
    model.backbone.model.set_grad_checkpointing(True)
    print("Gradient checkpointing enabled for DINOv2 backbone.")


for p in model.parameters():
    p.requires_grad = True

print("Backbone feature dim:", model.backbone.num_features)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))
print("Total parameters:", sum(p.numel() for p in model.parameters()))


pos_counts = np.zeros(n_labels, dtype=np.float32)
for inds in train_cls_df["label_indices"]:
    pos_counts[inds] += 1
pos_weight = (len(train_cls_df) - pos_counts) / np.maximum(pos_counts, 1)
pos_weight = np.clip(pos_weight, 1.0, 20.0)
pos_weight = torch.tensor(pos_weight, dtype=torch.float32, device=device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW([
    {"params": model.backbone.parameters(), "lr": CFG["backbone_lr"]},
    {"params": model.head.parameters(), "lr": CFG["head_lr"]},
], weight_decay=CFG["weight_decay"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, CFG["epochs"]))
scaler = torch.cuda.amp.GradScaler(enabled=(CFG["amp"] and device.type == "cuda"))
CKPT_PATH = os.path.join(OUT_DIR, f"best_{CFG['backbone_key']}_full_dinov2.pth")

train_loader = DataLoader(
    FullTuneCuiDataset(train_cls_df, train_tfms, n_labels),
    batch_size=CFG["batch_size"], shuffle=True,
    num_workers=CFG["num_workers"], pin_memory=True,
    drop_last=False,
)
valid_loader = DataLoader(
    FullTuneCuiDataset(valid_cls_df, eval_tfms, n_labels),
    batch_size=CFG["eval_batch_size"], shuffle=False,
    num_workers=CFG["num_workers"], pin_memory=True,
    drop_last=False,
)


def run_one_epoch(loader, train=True):
    model.train(train)
    total, n = 0.0, 0
    optimizer.zero_grad(set_to_none=True)

    for step, (imgs, y) in enumerate(loader, start=1):
        imgs = imgs.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        with torch.set_grad_enabled(train):
            with torch.cuda.amp.autocast(enabled=(CFG["amp"] and device.type == "cuda")):
                logits = model(imgs)
                loss = criterion(logits, y)
                loss_for_backward = loss / max(1, CFG["grad_accum_steps"])

            if train:
                scaler.scale(loss_for_backward).backward()
                should_step = (step % CFG["grad_accum_steps"] == 0) or (step == len(loader))
                if should_step:
                    if CFG["grad_clip_norm"] is not None:
                        scaler.unscale_(optimizer)
                        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip_norm"])
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)

        total += float(loss.item()) * imgs.size(0)
        n += imgs.size(0)

    return total / max(n, 1)

history = []
best_val = float("inf")

if CFG["train"]:
    print(f"\nFULL fine-tuning {CFG['model_name']} for {CFG['epochs']} epochs...")
    print("This updates the DINOv2 backbone + projection/classifier head.")
    for epoch in range(1, CFG["epochs"] + 1):
        t0 = time.time()
        tr_loss = run_one_epoch(train_loader, train=True)
        va_loss = run_one_epoch(valid_loader, train=False)
        scheduler.step()

        row = {
            "epoch": epoch,
            "train_loss": tr_loss,
            "valid_loss": va_loss,
            "backbone_lr": optimizer.param_groups[0]["lr"],
            "head_lr": optimizer.param_groups[1]["lr"],
            "seconds": time.time() - t0,
        }
        history.append(row)
        print(
            f"Epoch {epoch:02d}/{CFG['epochs']} | train={tr_loss:.5f} | valid={va_loss:.5f} | "
            f"bb_lr={row['backbone_lr']:.2e} | head_lr={row['head_lr']:.2e} | time={row['seconds']:.1f}s"
        )

        if va_loss < best_val:
            best_val = va_loss
            torch.save({
                "model": model.state_dict(),
                "cfg": CFG,
                "label_to_idx": label_to_idx,
                "best_val": best_val,
                "backbone_num_features": model.backbone.num_features,
            }, CKPT_PATH)
            print("  saved best full-model checkpoint")

    hist_df = pd.DataFrame(history)
    hist_df.to_csv(os.path.join(OUT_DIR, "training_history.csv"), index=False)

    plt.figure(figsize=(6,4))
    plt.plot(hist_df["epoch"], hist_df["train_loss"], marker="o", label="train")
    plt.plot(hist_df["epoch"], hist_df["valid_loss"], marker="o", label="valid")
    plt.title("Full DINOv2 fine-tuning loss")
    plt.xlabel("Epoch")
    plt.ylabel("BCE loss")
    plt.legend()
    savefig("04_training_loss_curve.png")

if os.path.exists(CKPT_PATH):
    ckpt = torch.load(CKPT_PATH, map_location=device)
    model.load_state_dict(ckpt["model"])
    print("Loaded best full-model checkpoint:", CKPT_PATH)


@torch.no_grad()
def extract_final_embeddings(df, split_name):
    loader = DataLoader(
        RocoImageOnlyDataset(df, eval_tfms),
        batch_size=CFG["eval_batch_size"], shuffle=False,
        num_workers=CFG["num_workers"], pin_memory=True,
    )
    model.eval()
    embs, ids = [], []
    for imgs, image_ids in loader:
        imgs = imgs.to(device, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=(CFG["amp"] and device.type == "cuda")):
            z = model(imgs, return_embedding=True)
        embs.append(z.float().cpu().numpy().astype("float32"))
        ids.extend(list(image_ids))
    embs = np.concatenate(embs, axis=0).astype("float32")
    np.save(os.path.join(OUT_DIR, f"{split_name}_embeddings.npy"), embs)
    pd.DataFrame({"image_id": ids}).to_csv(os.path.join(OUT_DIR, f"{split_name}_ids.csv"), index=False)
    print(split_name, embs.shape)
    return embs, ids

split_to_df = {"train": train_df, "valid": valid_df, "test": test_df}
query_df = split_to_df[CFG["eval_split"]].reset_index(drop=True)
pool_df = split_to_df[CFG["retrieval_pool"]].reset_index(drop=True)

query_embs, query_ids = extract_final_embeddings(query_df, CFG["eval_split"] + "_query")
if CFG["eval_split"] == CFG["retrieval_pool"] and len(query_df) == len(pool_df) and list(query_df["image_id"]) == list(pool_df["image_id"]):
    pool_embs, pool_ids = query_embs, query_ids
else:
    pool_embs, pool_ids = extract_final_embeddings(pool_df, CFG["retrieval_pool"] + "_pool")

same_pool = (
    CFG["eval_split"] == CFG["retrieval_pool"] and
    len(query_df) == len(pool_df) and
    list(query_df["image_id"]) == list(pool_df["image_id"])
)
print("same query/pool:", same_pool)


def compute_topk_by_embedding(query_embs, pool_embs, topk, chunk_size, same_pool):
    q = torch.tensor(query_embs, dtype=torch.float32, device=device)
    p = torch.tensor(pool_embs, dtype=torch.float32, device=device)
    all_idx, all_sim = [], []

    for s in range(0, q.shape[0], chunk_size):
        e = min(s + chunk_size, q.shape[0])
        sim = q[s:e] @ p.T

        if same_pool:
            diag = torch.arange(s, e, device=device)
            sim[torch.arange(e - s, device=device), diag] = -1e9

        vals, inds = torch.topk(sim, k=min(topk, p.shape[0] - 1 if same_pool else p.shape[0]), dim=1)
        all_idx.append(inds.cpu().numpy())
        all_sim.append(vals.cpu().numpy())

    return np.vstack(all_idx), np.vstack(all_sim)

topk_idx, topk_sim = compute_topk_by_embedding(
    query_embs, pool_embs,
    topk=CFG["topk_max"],
    chunk_size=CFG["eval_chunk_size"],
    same_pool=same_pool,
)

np.save(os.path.join(OUT_DIR, "retrieval_topk_indices.npy"), topk_idx)
np.save(os.path.join(OUT_DIR, "retrieval_topk_similarities.npy"), topk_sim)

rows = []
for qi in range(len(query_df)):
    for r in range(topk_idx.shape[1]):
        pi = int(topk_idx[qi, r])
        rows.append({
            "query_image_id": query_df.iloc[qi]["image_id"],
            "rank": r + 1,
            "retrieved_image_id": pool_df.iloc[pi]["image_id"],
            "cosine_similarity": float(topk_sim[qi, r]),
        })
pd.DataFrame(rows).to_csv(os.path.join(OUT_DIR, "retrieval_top50.csv"), index=False)


def build_cui_vocab(*dfs):
    vocab = {}
    for df in dfs:
        for cuis in df["cuis"]:
            for c in cuis:
                if c not in vocab:
                    vocab[c] = len(vocab)
    return vocab

def build_csr(df, vocab):
    rows, cols = [], []
    for i, cuis in enumerate(df["cuis"]):
        for c in set(cuis):
            if c in vocab:
                rows.append(i)
                cols.append(vocab[c])
    data = np.ones(len(rows), dtype=np.float32)
    mat = csr_matrix((data, (rows, cols)), shape=(len(df), len(vocab)), dtype=np.float32)
    lengths = np.asarray(mat.sum(axis=1)).reshape(-1).astype(np.float32)
    return mat, lengths

cui_vocab = build_cui_vocab(query_df, pool_df)
query_csr, query_len = build_csr(query_df, cui_vocab)
pool_csr, pool_len = build_csr(pool_df, cui_vocab)
print("CUI vocab:", len(cui_vocab))


def dcg_at_k(gains):
    gains = np.asarray(gains, dtype=np.float32)
    if gains.size == 0:
        return 0.0
    discounts = 1.0 / np.log2(np.arange(2, gains.size + 2))
    return float(np.sum(gains * discounts))

def ap_at_k(rel, total_relevant, k):
    rel = np.asarray(rel[:k], dtype=bool)
    if total_relevant <= 0:
        return np.nan
    hits = 0
    score = 0.0
    for i, r in enumerate(rel, start=1):
        if r:
            hits += 1
            score += hits / i
    return float(score / min(total_relevant, k))

def rr_at_k(rel, k):
    rel = np.asarray(rel[:k], dtype=bool)
    hit = np.where(rel)[0]
    return float(1.0 / (hit[0] + 1)) if len(hit) else 0.0

def summarize(values):
    arr = np.asarray(values, dtype=np.float64)
    arr = arr[~np.isnan(arr)]
    n = len(arr)
    if n == 0:
        return np.nan, np.nan, np.nan, 0
    mean = float(arr.mean())
    std = float(arr.std(ddof=1)) if n > 1 else 0.0
    ci95 = 1.96 * std / math.sqrt(n) if n > 1 else 0.0
    return mean, std, ci95, n

def evaluate_metrics(query_csr, pool_csr, query_len, pool_len, pred_idx, ks, threshold, same_pool, chunk_size):
    maxk = max(ks)
    per_query = []

    for s in range(0, query_csr.shape[0], chunk_size):
        e = min(s + chunk_size, query_csr.shape[0])

        inter = query_csr[s:e].dot(pool_csr.T).toarray().astype(np.float32)
        union = query_len[s:e, None] + pool_len[None, :] - inter
        all_gains = np.divide(inter, np.maximum(union, 1e-8), out=np.zeros_like(inter), where=union > 0)

        if same_pool:
            for local_i, global_i in enumerate(range(s, e)):
                all_gains[local_i, global_i] = -1.0

        for local_i, global_i in enumerate(range(s, e)):
            top_idx = pred_idx[global_i, :maxk]
            pred_gains = all_gains[local_i, top_idx].copy()
            pred_gains[pred_gains < 0] = 0.0

            gains = all_gains[local_i].copy()
            gains[gains < 0] = 0.0
            total_relevant = int((gains > threshold).sum())
            ideal_sorted = np.sort(gains)[::-1]

            row = {
                "query_index": global_i,
                "query_image_id": query_df.iloc[global_i]["image_id"],
                "total_relevant_by_cui_jaccard": total_relevant,
            }

            for k in ks:
                gk = pred_gains[:k]
                rel = gk > threshold

                idcg = dcg_at_k(ideal_sorted[:k])
                row[f"CUI@{k}"] = dcg_at_k(gk) / idcg if idcg > 0 else 0.0
                row[f"Precision@{k}"] = float(rel.sum() / k)
                row[f"Recall@{k}"] = float(rel.sum() / total_relevant) if total_relevant > 0 else np.nan
                row[f"mAP@{k}"] = ap_at_k(rel, total_relevant, k)
                row[f"MRR@{k}"] = rr_at_k(rel, k)
                row[f"MeanJaccard@{k}"] = float(np.mean(gk))

            per_query.append(row)

    per_query = pd.DataFrame(per_query)

    summary = []
    for col in [c for c in per_query.columns if "@" in c]:
        mean, std, ci95, n = summarize(per_query[col].values)
        summary.append({
            "metric": col,
            "mean": mean,
            "std": std,
            "ci95": ci95,
            "n_queries_used": n,
            "mean±std": f"{mean:.4f} ± {std:.4f}",
            "mean±95%CI": f"{mean:.4f} ± {ci95:.4f}",
        })

    return per_query, pd.DataFrame(summary)

all_ks = sorted(set(CFG["ks_main"] + CFG["ks_curve"]))
per_query_metrics, metrics_summary = evaluate_metrics(
    query_csr, pool_csr, query_len, pool_len,
    topk_idx, all_ks,
    threshold=CFG["relevance_threshold"],
    same_pool=same_pool,
    chunk_size=CFG["eval_chunk_size"],
)

per_query_metrics.to_csv(os.path.join(OUT_DIR, "per_query_metrics.csv"), index=False)
metrics_summary.to_csv(os.path.join(OUT_DIR, "metrics_summary_all_k.csv"), index=False)

main_cols = []
for metric in ["CUI", "Precision", "Recall", "mAP", "MRR", "MeanJaccard"]:
    for k in CFG["ks_main"]:
        main_cols.append(f"{metric}@{k}")

main_summary = metrics_summary[metrics_summary["metric"].isin(main_cols)].copy()
main_summary["order"] = main_summary["metric"].apply(lambda x: main_cols.index(x))
main_summary = main_summary.sort_values("order").drop(columns=["order"])
main_summary.to_csv(os.path.join(OUT_DIR, "metrics_summary_main.csv"), index=False)

print("\n========== MAIN RESULTS ==========")
display(main_summary[["metric", "mean±std", "mean±95%CI", "n_queries_used"]])

no_rel = int((per_query_metrics["total_relevant_by_cui_jaccard"] == 0).sum())
print(f"Queries with no relevant non-self image: {no_rel}/{len(per_query_metrics)}")


def metric_curve(summary, prefix):
    xs, means, cis = [], [], []
    for k in CFG["ks_curve"]:
        name = f"{prefix}@{k}"
        hit = summary[summary["metric"] == name]
        if len(hit):
            xs.append(k)
            means.append(float(hit["mean"].iloc[0]))
            cis.append(float(hit["ci95"].iloc[0]))
    return np.array(xs), np.array(means), np.array(cis)

for metric in ["CUI", "Precision", "Recall", "mAP", "MRR", "MeanJaccard"]:
    xs, means, cis = metric_curve(metrics_summary, metric)
    if len(xs):
        plt.figure(figsize=(6,4))
        plt.plot(xs, means, marker="o", markersize=3)
        plt.fill_between(xs, means-cis, means+cis, alpha=0.2)
        plt.title(f"{metric}@K")
        plt.xlabel("K")
        plt.ylabel(metric)
        savefig(f"05_{metric.lower()}_at_k_curve.png")


p_x, p_mean, _ = metric_curve(metrics_summary, "Precision")
r_x, r_mean, _ = metric_curve(metrics_summary, "Recall")
if len(p_mean) and len(r_mean):
    L = min(len(p_mean), len(r_mean))
    plt.figure(figsize=(5,5))
    plt.plot(r_mean[:L], p_mean[:L], marker="o", markersize=3)
    plt.title("Precision-Recall curve across K")
    plt.xlabel("Recall@K")
    plt.ylabel("Precision@K")
    savefig("06_precision_recall_curve_across_k.png")


rng = np.random.default_rng(CFG["seed"])
sample_n = min(2000, len(query_embs))
if sample_n >= 10:
    sample_idx = rng.choice(len(query_embs), size=sample_n, replace=False)
    pca = PCA(n_components=2, random_state=CFG["seed"])
    xy = pca.fit_transform(query_embs[sample_idx])
    kmeans_k = min(10, sample_n)
    labels = KMeans(n_clusters=kmeans_k, random_state=CFG["seed"], n_init=10).fit_predict(xy)

    plt.figure(figsize=(6,5))
    plt.scatter(xy[:,0], xy[:,1], c=labels, s=8, alpha=0.75)
    plt.title("PCA of query image embeddings")
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    savefig("07_embedding_pca_kmeans.png")

plt.figure(figsize=(6,4))
plt.hist(topk_sim[:, 0], bins=40, alpha=0.8)
plt.title("Top-1 cosine similarity distribution")
plt.xlabel("Cosine similarity")
plt.ylabel("Queries")
savefig("08_top1_similarity_histogram.png")

plt.figure(figsize=(6,4))
plt.hist(topk_sim.reshape(-1), bins=60, alpha=0.8)
plt.title("Top-K cosine similarity distribution")
plt.xlabel("Cosine similarity")
plt.ylabel("Retrieved pairs")
savefig("09_topk_similarity_histogram.png")


def load_img_for_plot(path):
    return Image.open(path).convert("RGB")

def jaccard(a, b):
    a, b = set(a), set(b)
    if not a and not b:
        return 0.0
    return len(a & b) / max(len(a | b), 1)

def plot_retrieval_grid(query_indices, k, name):
    if len(query_indices) == 0:
        return
    n_rows = len(query_indices)
    n_cols = k + 1
    plt.figure(figsize=(2.1*n_cols, 2.3*n_rows))

    for rr, qi in enumerate(query_indices):
        qrow = query_df.iloc[qi]
        ax = plt.subplot(n_rows, n_cols, rr*n_cols + 1)
        ax.imshow(load_img_for_plot(qrow["path"]))
        ax.set_title("Query\n" + str(qrow["image_id"])[-10:], fontsize=8)
        ax.axis("off")

        for j in range(k):
            pi = int(topk_idx[qi, j])
            prow = pool_df.iloc[pi]
            ax = plt.subplot(n_rows, n_cols, rr*n_cols + 2 + j)
            ax.imshow(load_img_for_plot(prow["path"]))
            jac = jaccard(qrow["cuis"], prow["cuis"])
            ax.set_title(f"Rank {j+1}\nJ={jac:.2f}", fontsize=8)
            ax.axis("off")

    savefig(name)

rand_q = rng.choice(len(query_df), size=min(6, len(query_df)), replace=False).tolist()
plot_retrieval_grid(rand_q, 5, "10_random_top5_retrieval_examples.png")

if "CUI@5" in per_query_metrics.columns:
    best_q = per_query_metrics.sort_values("CUI@5", ascending=False)["query_index"].head(6).tolist()
    worst_q = per_query_metrics.sort_values("CUI@5", ascending=True)["query_index"].head(6).tolist()
    plot_retrieval_grid(best_q, 5, "11_best_cui5_top5_retrieval_examples.png")
    plot_retrieval_grid(worst_q, 5, "12_worst_cui5_top5_retrieval_examples.png")


with open(os.path.join(OUT_DIR, "config.json"), "w") as f:
    json.dump(CFG, f, indent=2)

zip_path = os.path.join(os.environ.get("OUTPUT_ROOT", "../results"), "dinov2_roco_full_finetune_retrieval_figures.zip")
if os.path.exists(zip_path + ".zip"):
    os.remove(zip_path + ".zip")
shutil.make_archive(zip_path, "zip", FIG_DIR)

print("\n========== SAVED FILES ==========")
for root, dirs, files in os.walk(OUT_DIR):
    level = root.replace(OUT_DIR, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in sorted(files):
        print(f"{'  ' * (level + 1)}{f}")

print("\nDone.")
print("Main metrics:", os.path.join(OUT_DIR, "metrics_summary_main.csv"))
print("All K metrics:", os.path.join(OUT_DIR, "metrics_summary_all_k.csv"))
print("Figures:", FIG_DIR)
print("Figures zip:", zip_path + ".zip")
